In [2]:
import pygame
import random
import math

pygame.init()

pygame.mixer.init()

click_sound = pygame.mixer.Sound("click.mp3")
mine_sound = pygame.mixer.Sound("boom.mp3")
victory_sound = pygame.mixer.Sound("victory.mp3")
start_sound = pygame.mixer.Sound("start.mp3") 

# ==========================
# CONFIG
# ==========================
ROWS = 10
COLS = 10
MINES = 15

CELL_SIZE = 50
TOP_BAR = 120

WIDTH = COLS * CELL_SIZE
HEIGHT = ROWS * CELL_SIZE + TOP_BAR

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Minesweeper")

clock = pygame.time.Clock()

# ==========================
# COLORS
# ==========================
BG = (18, 18, 30)

HIDDEN = (55, 65, 81)
HOVER = (75, 85, 101)

REVEALED = (229, 231, 235)

GRID = (31, 41, 55)

FLAG_COLOR = (245, 158, 11)
MINE_COLOR = (239, 68, 68)

WHITE = (255, 255, 255)

NUMBER_COLORS = {
    1: (59,130,246),
    2: (34,197,94),
    3: (239,68,68),
    4: (168,85,247),
    5: (220,38,38),
    6: (6,182,212),
    7: (0,0,0),
    8: (100,100,100)
}

# ==========================
# FONTS
# ==========================
font = pygame.font.SysFont("Segoe UI", 24)
big_font = pygame.font.SysFont("Segoe UI", 48, bold=True)

# ==========================
# GAME STATE
# ==========================
board = []
revealed = []
flagged = []

flags_left = MINES
game_over = False
won = False


# ==========================
# TEXT
# ==========================
def draw_centered_text(text, font, color, y):

    shadow = font.render(text, True, (0,0,0))
    main = font.render(text, True, color)

    x = WIDTH//2 - main.get_width()//2

    screen.blit(shadow, (x+3, y+3))
    screen.blit(main, (x, y))


# ==========================
# RESET GAME
# ==========================
def reset_game():

    global board
    global revealed
    global flagged

    global flags_left
    global game_over
    global won

    flags_left = MINES
    game_over = False
    won = False

    board = [[0]*COLS for _ in range(ROWS)]
    revealed = [[False]*COLS for _ in range(ROWS)]
    flagged = [[False]*COLS for _ in range(ROWS)]

    mines = random.sample(
        [(r,c) for r in range(ROWS) for c in range(COLS)],
        MINES
    )

    for r,c in mines:
        board[r][c] = -1

    for r in range(ROWS):
        for c in range(COLS):

            if board[r][c] == -1:
                continue

            count = 0

            for dr in (-1,0,1):
                for dc in (-1,0,1):

                    nr = r + dr
                    nc = c + dc

                    if 0 <= nr < ROWS and 0 <= nc < COLS:
                        if board[nr][nc] == -1:
                            count += 1

            board[r][c] = count


# ==========================
# REVEAL
# ==========================
def reveal(r,c):

    if not (0 <= r < ROWS and 0 <= c < COLS):
        return

    if revealed[r][c]:
        return

    if flagged[r][c]:
        return

    revealed[r][c] = True

    if board[r][c] == 0:

        for dr in (-1,0,1):
            for dc in (-1,0,1):

                if dr == 0 and dc == 0:
                    continue

                reveal(r+dr,c+dc)


# ==========================
# WIN CHECK
# ==========================
def check_win():

    for r in range(ROWS):
        for c in range(COLS):

            if board[r][c] != -1 and not revealed[r][c]:
                return False

    return True


# ==========================
# INITIALIZE
# ==========================
reset_game()
start_sound.play()

restart_button = pygame.Rect(
    WIDTH - 140,
    25,
    120,
    45
)

# ==========================
# MAIN LOOP
# ==========================
running = True
victory_played = False

while running:

    clock.tick(60)

    mx,my = pygame.mouse.get_pos()

    # Gradient Background
    for y in range(HEIGHT):

        shade = min(255, 18 + y//15)

        pygame.draw.line(
            screen,
            (shade, shade, shade+10),
            (0,y),
            (WIDTH,y)
        )

    # ----------------------
    # EVENTS
    # ----------------------
    for event in pygame.event.get():

        if event.type == pygame.QUIT:
            running = False

        if event.type == pygame.MOUSEBUTTONDOWN:

            if restart_button.collidepoint(event.pos):
                reset_game()
                victory_played = False
                start_sound.play()
                continue
                
            if game_over or won:
                continue

            mx,my = event.pos

            if my < TOP_BAR:
                continue

            row = (my - TOP_BAR)//CELL_SIZE
            col = mx//CELL_SIZE

            # Left Click
            if event.button == 1:

                if board[row][col] == -1:
                    mine_sound.play()
                    game_over = True
                    for r in range(ROWS):
                        for c in range(COLS):
                            revealed[r][c] = True

                else:
                    
                    click_sound.play() 
                    reveal(row,col)

            # Right Click
            elif event.button == 3:

                if not revealed[row][col]:

                    if flagged[row][col]:

                        flagged[row][col] = False
                        flags_left += 1

                    elif flags_left > 0:

                        flagged[row][col] = True
                        flags_left -= 1
                        
    won = check_win() if not game_over else False
    if won and not victory_played:
        victory_sound.play()
        victory_played = True

    # ----------------------
    # TOP BAR
    # ----------------------
    pygame.draw.rect(
        screen,
        (15,15,25),
        (0,0,WIDTH,TOP_BAR)
    )

    pygame.draw.rect(
        screen,
        (40,40,60),
        (15,20,170,50),
        border_radius=12
    )

    flags_text = font.render(
        f"Flags: {flags_left}",
        True,
        WHITE
    )

    screen.blit(flags_text,(30,32))

    pygame.draw.rect(
        screen,
        (59,130,246),
        restart_button,
        border_radius=12
    )

    restart_text = font.render(
        "Restart",
        True,
        WHITE
    )

    screen.blit(
        restart_text,
        (
            restart_button.centerx -
            restart_text.get_width()//2,
            restart_button.centery -
            restart_text.get_height()//2
        )
    )

    # ----------------------
    # STATUS
    # ----------------------


    if game_over:

        status_font = pygame.font.SysFont(
            "Segoe UI",
            28,
            bold=True
        )

        draw_centered_text(
            "GAME OVER",
            status_font,
            (255,80,80),
            35
        )

    elif won:

        pulse = 48 + int(
            4 * abs(math.sin(
                pygame.time.get_ticks()/300
            ))
        )

        status_font = pygame.font.SysFont(
            "Segoe UI",
            32,
            bold=True
        )

        draw_centered_text(
            "YOU WIN!",
            status_font,
            (50,220,50),
            35
        )

    # ----------------------
    # BOARD
    # ----------------------
    for r in range(ROWS):
        for c in range(COLS):

            rect = pygame.Rect(
                c*CELL_SIZE,
                TOP_BAR + r*CELL_SIZE,
                CELL_SIZE,
                CELL_SIZE
            )

            if rect.collidepoint(mx,my) and not revealed[r][c]:
                cell_color = HOVER
            else:
                cell_color = HIDDEN

            if revealed[r][c]:

                pygame.draw.rect(
                    screen,
                    REVEALED,
                    rect,
                    border_radius=10
                )

                if board[r][c] == -1:

                    pygame.draw.circle(
                        screen,
                        MINE_COLOR,
                        rect.center,
                        10
                    )

                elif board[r][c] > 0:

                    text = font.render(
                        str(board[r][c]),
                        True,
                        NUMBER_COLORS[board[r][c]]
                    )

                    screen.blit(
                        text,
                        (
                            rect.centerx - text.get_width()//2,
                            rect.centery - text.get_height()//2
                        )
                    )

            else:

                pygame.draw.rect(
                    screen,
                    cell_color,
                    rect,
                    border_radius=10
                )

                if flagged[r][c]:

                    pygame.draw.polygon(
                        screen,
                        FLAG_COLOR,
                        [
                            (rect.x+15,rect.y+10),
                            (rect.x+35,rect.y+18),
                            (rect.x+15,rect.y+26)
                        ]
                    )

                    pygame.draw.line(
                        screen,
                        WHITE,
                        (rect.x+15,rect.y+10),
                        (rect.x+15,rect.y+35),
                        3
                    )

            pygame.draw.rect(
                screen,
                GRID,
                rect,
                1,
                border_radius=10
            )

    pygame.display.flip()

pygame.quit()